In [2]:
import json
from collections import defaultdict
    
with open('data/reddit_comment_body_dec_2024.json', 'r') as f:
    data = json.load(f)
    

# Create a dictionary to store messages by author
author_messages = defaultdict(list)
for item in data:
    author_messages[item['author']].append(item['body'])

2 Author Dataset

In [3]:
import random

# Get eligible authors (with at least two messages)
eligible_authors = [auth for auth, msgs in author_messages.items() if len(msgs) >= 2]
if len(eligible_authors) < 2:
    raise ValueError("Not enough authors with at least 2 messages.")

# Randomly choose 2 authors
author1, author2 = random.sample(eligible_authors, 2)

def create_positive_pairs(msgs, n_pairs):
    msgs_copy = msgs[:]  # copy to avoid modifying original list
    random.shuffle(msgs_copy)
    pairs = []
    for i in range(n_pairs):
        pairs.append((msgs_copy[2 * i], msgs_copy[2 * i + 1]))
    return pairs

# Determine how many pairs we can form from each author so they're balanced.
n_pairs_author1 = len(author_messages[author1]) // 2
n_pairs_author2 = len(author_messages[author2]) // 2
n_pairs = min(n_pairs_author1, n_pairs_author2)
if n_pairs == 0:
    raise ValueError("Not enough messages to form pairs for both authors.")

# Create positive examples (same-author pairs with label 1)
pos_pairs_auth1 = create_positive_pairs(author_messages[author1], n_pairs)
pos_pairs_auth2 = create_positive_pairs(author_messages[author2], n_pairs)

training_examples = []
for pair in pos_pairs_auth1:
    training_examples.append((pair[0], pair[1], 1, author1, author1))
for pair in pos_pairs_auth2:
    training_examples.append((pair[0], pair[1], 1, author2, author2))

# For negative examples (different-author pairs with label 0), 
# use the messages selected in the positive pairing to balance the examples.
auth1_msgs = [msg for pair in pos_pairs_auth1 for msg in pair]
auth2_msgs = [msg for pair in pos_pairs_auth2 for msg in pair]

random.shuffle(auth1_msgs)
random.shuffle(auth2_msgs)

n_negative = min(len(auth1_msgs), len(auth2_msgs))
for i in range(n_negative):
    training_examples.append((auth1_msgs[i], auth2_msgs[i], 0, author1, author2))

# Shuffle training examples
random.shuffle(training_examples)

# Check balance and sample output
pos_count = sum(1 for ex in training_examples if ex[2] == 1)
neg_count = sum(1 for ex in training_examples if ex[2] == 0)
print("Selected authors:", author1, author2)
print("Number of positive examples:", pos_count)
print("Number of negative examples:", neg_count)
print("Sample training examples:")
for ex in training_examples[:5]:
    print(ex)

Selected authors: king-balls1 Scams-ModTeam
Number of positive examples: 4042
Number of negative examples: 4042
Sample training examples:
('Dawg the person he\'s talking about has said some wild stuff like "underage females are sex slaves" And something about him wanting to be away from flat women LIKE HE\'S LITERALLY HATING ON KIDS AND MOTHERS.', "*Your submission was manually removed by a moderator for the following reason:*\n\n**Subreddit Rule 4: Spam or joke**\n\nThis subreddit is a place for useful and informative discussions about scams. We do not allow:\n\n* Unhelpful content\n* Jokes on serious posts\n* Sarcasm, even if obvious or tagged, since it can be construed as harmful advice\n* Anything not related to the scam being discussed\n\nPlease keep content submitted to this subreddit useful, relevant and meaningful.\n\nBefore posting again, make sure you review the [rules of our subreddit.](https://www.reddit.com/r/Scams/wiki/rules/)\n\n^(If you believe this is a mistake, feel f

In [4]:
import torch
from transformers import BertModel
import torch.nn as nn

class SiameseBERT(nn.Module):
    def __init__(self, pretrained_model_name="bert-base-uncased", hidden_size=768, dropout_prob=0.1):
        super(SiameseBERT, self).__init__()
        self.bert = BertModel.from_pretrained(pretrained_model_name)
        self.dropout = nn.Dropout(dropout_prob)
        
        # Add additional fully connected layers after BERT
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout_prob),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU()
        )
        
        # Updated classifier accepting reduced dimension
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size // 2, 1),
            nn.Sigmoid()
        )
        
    def forward(self, input_ids1, attention_mask1, input_ids2, attention_mask2):
        out1 = self.bert(input_ids=input_ids1, attention_mask=attention_mask1)
        embed1 = out1.pooler_output  # [batch_size, hidden_size]
        embed1 = self.dropout(embed1)
        embed1 = self.fc(embed1)
        
        out2 = self.bert(input_ids=input_ids2, attention_mask=attention_mask2)
        embed2 = out2.pooler_output  # [batch_size, hidden_size]
        embed2 = self.dropout(embed2)
        embed2 = self.fc(embed2)
        
        # Use the absolute difference of the refined embeddings
        diff = torch.abs(embed1 - embed2)
        prob = self.classifier(diff)
        return prob, embed1, embed2


def contrastive_loss(embedding1, embedding2, label, margin=1.0):
    """
    Computes the contrastive loss.
    label: 1 if same author, 0 otherwise.
    """
    # Calculate Euclidean distance between embeddings
    distance = torch.norm(embedding1 - embedding2, p=2, dim=1)
    # Contrastive loss from Hadsell et al.
    loss = label * torch.pow(distance, 2) + (1 - label) * torch.pow(torch.clamp(margin - distance, min=0.0), 2)
    return loss.mean()


In [5]:
from transformers import BertTokenizer
import torch

# Initialize tokenizer (if not already defined)
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Extract texts and labels from training_examples (already defined in a previous cell)
texts1 = [ex[0] for ex in training_examples]
texts2 = [ex[1] for ex in training_examples]
labels = [ex[2] for ex in training_examples]

# Tokenize each set of texts
encoded_inputs1 = tokenizer(texts1, padding=True, truncation=True, return_tensors="pt")
encoded_inputs2 = tokenizer(texts2, padding=True, truncation=True, return_tensors="pt")

# Convert labels to a tensor
labels = torch.tensor(labels, dtype=torch.float)

# Pack tokenized data into a dictionary
training_data = {
    "input_ids1": encoded_inputs1["input_ids"],
    "attention_mask1": encoded_inputs1["attention_mask"],
    "input_ids2": encoded_inputs2["input_ids"],
    "attention_mask2": encoded_inputs2["attention_mask"],
    "labels": labels
}

print("Tokenization complete. Example input_ids1 for first example:")
print(training_data["input_ids1"][0])

Tokenization complete. Example input_ids1 for first example:
tensor([  101,  4830, 27767,  1996,  2711,  2002,  1005,  1055,  3331,  2055,
         2038,  2056,  2070,  3748,  4933,  2066,  1000,  2104,  4270,  3801,
         2024,  3348,  7179,  1000,  1998,  2242,  2055,  2032,  5782,  2000,
         2022,  2185,  2013,  4257,  2308,  2066,  2002,  1005,  1055,  6719,
        22650,  2006,  4268,  1998, 10756,  1012,   102,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
   

In [ ]:
from sklearn.model_selection import KFold
from torch.utils.data import TensorDataset, DataLoader

import torch.nn.functional as F

from tqdm.notebook import tqdm
import numpy as np
from sklearn.decomposition import PCA

        
import matplotlib.pyplot as plt

debug = True

if debug:
    print("Starting training: Dataset sizes:")
    print("training_data['input_ids1'] shape:", training_data["input_ids1"].shape)
    print("training_data['attention_mask1'] shape:", training_data["attention_mask1"].shape)
    print("training_data['input_ids2'] shape:", training_data["input_ids2"].shape)
    print("training_data['attention_mask2'] shape:", training_data["attention_mask2"].shape)
    print("training_data['labels'] shape:", training_data["labels"].shape)


# Prepare dataset using tensors from training_data defined earlier
dataset = TensorDataset(
    training_data["input_ids1"],
    training_data["attention_mask1"],
    training_data["input_ids2"],
    training_data["attention_mask2"],
    training_data["labels"]
)

# Define device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Instantiate model and move to device
model = SiameseBERT().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
n_epochs = 3
batch_size = 8

# Freeze the Bert encoder so its parameters are not updated during training
for param in model.bert.encoder.parameters():
    param.requires_grad = False

kf = KFold(n_splits=5, shuffle=True, random_state=42)

fold = 0
for train_index, val_index in kf.split(dataset):
    fold += 1
    print(f"\nFold {fold}")
    
    train_subset = torch.utils.data.Subset(dataset, train_index)
    val_subset = torch.utils.data.Subset(dataset, val_index)
    
    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=batch_size)
    
    for epoch in range(1, n_epochs + 1):
        model.train()
        train_loss = 0
        correct = 0
        total = 0
        
        # Use tqdm progress bar on the training loop
        for batch in tqdm(train_loader, desc=f"Epoch {epoch}"):
            
            input_ids1, mask1, input_ids2, mask2, labels = batch
            input_ids1 = input_ids1.to(device)
            mask1 = mask1.to(device)
            input_ids2 = input_ids2.to(device)
            mask2 = mask2.to(device)
            labels = labels.to(device)
            
            if debug : print(f"batch size: {len(labels)}")
            
            optimizer.zero_grad()
            prob, emb1, emb2 = model(input_ids1, mask1, input_ids2, mask2)
            if debug : print(f"prob shape: {prob.shape}")
            if debug : print(f"emb1 shape: {emb1.shape}")
            if debug : print(f"first 50 probabilities: {prob.squeeze()[:50]}")
            
            loss = contrastive_loss(emb1, emb2, labels)
            if debug : print(f"loss: {loss}")
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * labels.size(0)
            # Compute accuracy (threshold probability at 0.5)
            preds = (prob.squeeze() > 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)
        
        avg_loss = train_loss / total
        train_acc = correct / total * 100
        print(f"Epoch {epoch} - Loss: {avg_loss:.4f} - Training Accuracy: {train_acc:.2f}%")



        # Visualize validation embeddings after the epoch
        model.eval()
        all_embeddings = []
        all_authors = []

        with torch.no_grad():
            # For each sample index in the current validation split (val_index)
            for idx in val_index:
                # Retrieve the tokenized inputs for both messages and add batch dimension
                sample_ids1 = training_data["input_ids1"][idx].unsqueeze(0).to(device)
                sample_mask1 = training_data["attention_mask1"][idx].unsqueeze(0).to(device)
                sample_ids2 = training_data["input_ids2"][idx].unsqueeze(0).to(device)
                sample_mask2 = training_data["attention_mask2"][idx].unsqueeze(0).to(device)
                
                # Get the embeddings for both messages (emb1 for text1, emb2 for text2)
                _, emb1, emb2 = model(sample_ids1, sample_mask1, sample_ids2, sample_mask2)
                all_embeddings.append(emb1.cpu())
                all_authors.append(training_examples[idx][3])  # original author for text1
                all_embeddings.append(emb2.cpu())
                all_authors.append(training_examples[idx][4])  # original author for text2

        # Concatenate all embeddings into a (num_messages x hidden_size) numpy array
        all_embeddings = torch.cat(all_embeddings, dim=0).numpy()

        # Reduce embeddings to 2 dimensions via PCA
        pca = PCA(n_components=2)
        embeddings_2d = pca.fit_transform(all_embeddings)

        # Plot each message embedding colored by its original author
        plt.figure(figsize=(8, 6))
        unique_authors = list(set(all_authors))
        colors = plt.cm.get_cmap("tab20", len(unique_authors))

        for i, author in enumerate(unique_authors):
            inds = [j for j, a in enumerate(all_authors) if a == author]
            plt.scatter(embeddings_2d[inds, 0], embeddings_2d[inds, 1],
                        color=colors(i), label=author, alpha=0.7)

        plt.title(f'Validation Embeddings at Epoch {epoch}')
        plt.xlabel('PCA Component 1')
        plt.ylabel('PCA Component 2')
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.show()
        
    # Validation after training fold
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids1, mask1, input_ids2, mask2, labels = batch
            input_ids1 = input_ids1.to(device)
            mask1 = mask1.to(device)
            input_ids2 = input_ids2.to(device)
            mask2 = mask2.to(device)
            labels = labels.to(device)
            
            prob, _, _ = model(input_ids1, mask1, input_ids2, mask2)
            preds = (prob.squeeze() > 0.5).float()
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
    val_acc = val_correct / val_total * 100
    print(f"Fold {fold} Validation Accuracy: {val_acc:.2f}%")


Starting training: Dataset sizes:
training_data['input_ids1'] shape: torch.Size([8084, 512])
training_data['attention_mask1'] shape: torch.Size([8084, 512])
training_data['input_ids2'] shape: torch.Size([8084, 512])
training_data['attention_mask2'] shape: torch.Size([8084, 512])
training_data['labels'] shape: torch.Size([8084])

Fold 1


Epoch 1:   0%|          | 0/809 [00:00<?, ?it/s]

batch size: 5
prob shape: torch.Size([8, 1])
emb1 shape: torch.Size([8, 768])
first 50 probabilities: tensor([0.4458, 0.5372, 0.3964, 0.4993, 0.4879, 0.4477, 0.4856, 0.4732],
       grad_fn=<SliceBackward0>)
loss: 66.593505859375
batch size: 5
prob shape: torch.Size([8, 1])
emb1 shape: torch.Size([8, 768])
first 50 probabilities: tensor([0.4205, 0.4977, 0.4482, 0.4732, 0.5289, 0.5367, 0.4719, 0.5026],
       grad_fn=<SliceBackward0>)
loss: 50.60320281982422
batch size: 5
prob shape: torch.Size([8, 1])
emb1 shape: torch.Size([8, 768])
first 50 probabilities: tensor([0.4598, 0.3910, 0.4968, 0.4410, 0.4936, 0.4746, 0.4516, 0.4161],
       grad_fn=<SliceBackward0>)
loss: 51.95701599121094
batch size: 5
prob shape: torch.Size([8, 1])
emb1 shape: torch.Size([8, 768])
first 50 probabilities: tensor([0.5263, 0.4997, 0.4378, 0.5784, 0.5141, 0.4960, 0.4850, 0.3983],
       grad_fn=<SliceBackward0>)
loss: 23.739604949951172
batch size: 5
prob shape: torch.Size([8, 1])
emb1 shape: torch.Size([8, 7

KeyboardInterrupt: 